# AI-Powered Australian Medical Practice Data Enrichment

**Project objective:** Collect Australian medical practice information from the Family Doctor clinic directory, clean and validate the source data, and combine it with evidence-based AI enrichment for `service_categories` and `telehealth`.

**Final dataset:** 86 unique medical practices.

> **Important:** The original source fields are preserved. AI enrichment must be supported by the information provided for the same clinic and must not rely on assumptions.

## 1. Install and import required libraries

Only the libraries required for this notebook are used:
- Selenium — browser-based web scraping
- BeautifulSoup — HTML parsing
- pandas — data cleaning and export

In [ ]:
# Run this cell once if the required packages are not installed.
%pip install -q selenium beautifulsoup4 pandas

In [ ]:
import json
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

## 2. Set up Selenium

Selenium is used because direct HTTP requests to the website were not reliable during data collection.

In [ ]:
# Start Chrome through Selenium.
options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)
print("Selenium browser started successfully.")

## 3. Scrape one clinic page

The clinic pages contain structured JSON-LD information. The scraper extracts:
- Practice name
- Phone
- Full address
- Opening hours
- Raw service information
- Source URL

The raw service text is preserved because it is the evidence used later for AI enrichment.

In [ ]:
def scrape_clinic(url):
    """Extract the required fields from one Family Doctor clinic page."""

    driver.get(url)
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    json_ld_blocks = soup.find_all("script", type="application/ld+json")

    faq_data = None
    graph_data = None

    for script in json_ld_blocks:
        try:
            data = json.loads(script.get_text(strip=True))

            if isinstance(data, dict) and data.get("@type") == "FAQPage":
                faq_data = data

            elif isinstance(data, dict) and "@graph" in data:
                graph_data = data["@graph"]

        except (json.JSONDecodeError, TypeError):
            continue

    # Extract practice name from structured page metadata.
    practice_name = ""
    if graph_data:
        for item in graph_data:
            if item.get("@type") == "WebPage":
                practice_name = item.get("name", "").split("|")[0].strip()
                break

    phone = ""
    full_address = ""
    opening_hours = ""
    services_raw = ""

    # Extract clinic information from FAQ structured data.
    if faq_data:
        for item in faq_data.get("mainEntity", []):
            question = item.get("name", "").lower()
            answer = item.get("acceptedAnswer", {}).get("text", "")

            if "opening hours" in question:
                opening_hours = answer

            elif "where is" in question:
                full_address = answer

            elif "how do i contact" in question:
                phone_match = re.search(
                    r'\(\d{2}\)\s?\d{4}\s?\d{4}',
                    answer
                )
                if phone_match:
                    phone = phone_match.group()

            elif "what services" in question:
                services_raw = answer

    # Clean common address wording.
    if "located at" in full_address.lower():
        full_address = re.split(
            r"located at",
            full_address,
            maxsplit=1,
            flags=re.IGNORECASE
        )[-1].strip()

    full_address = full_address.rstrip(".")

    # Fallback: read a telephone link when the FAQ does not contain a phone number.
    if not phone:
        tel_link = soup.find("a", href=re.compile(r"^tel:", re.I))
        if tel_link:
            phone = tel_link.get("href", "").replace("tel:", "").strip()

    return {
        "practice_name": practice_name,
        "phone": phone,
        "full_address": full_address,
        "opening_hours": opening_hours,
        "services_raw": services_raw,
        "source_url": url
    }

## 4. Collect clinic URLs

The Family Doctor directory contains clinic links as well as general information and location/SEO pages. The filtering below keeps likely clinic pages and removes known non-clinic pages.

In [ ]:
directory_url = "https://familydoctor.com.au/view-our-clinics/"

driver.get(directory_url)
time.sleep(3)

soup = BeautifulSoup(driver.page_source, "html.parser")

excluded_urls = {
    "https://familydoctor.com.au/",
    "https://familydoctor.com.au/view-our-clinics/",
    "https://familydoctor.com.au/health-advice/",
    "https://familydoctor.com.au/opportunities-with-family-doctor/",
    "https://familydoctor.com.au/gp/",
    "https://familydoctor.com.au/fsp/",
    "https://familydoctor.com.au/selling-your-practice/",
    "https://familydoctor.com.au/bulk-billing/",
    "https://familydoctor.com.au/allied-health-jobs/",
}

clinic_keywords = (
    "medical", "clinic", "health", "doctor",
    "gp", "practice", "surgery"
)

clinic_urls = []

for a in soup.find_all("a", href=True):
    name = a.get_text(" ", strip=True)
    href = a["href"].strip()

    if not name:
        continue

    if "familydoctor.com.au/" not in href:
        continue

    if href in excluded_urls:
        continue

    # Exclude location/SEO pages that are not individual clinic records.
    if "/gp-in-" in href:
        continue

    if any(keyword in name.lower() for keyword in clinic_keywords):
        clinic_urls.append(href)

clinic_urls = list(dict.fromkeys(clinic_urls))

print("Clinic URLs found:", len(clinic_urls))

## 5. Scrape the clinic pages

Each clinic is processed independently. Failed pages are recorded instead of stopping the complete scraping process.

In [ ]:
all_results = []
failed_urls = []

for i, url in enumerate(clinic_urls, start=1):
    print(f"[{i}/{len(clinic_urls)}] {url}")

    try:
        all_results.append(scrape_clinic(url))
        print("   ✓ Success")
    except Exception as exc:
        failed_urls.append({"url": url, "error": str(exc)})
        print("   ✗ Failed:", exc)

    time.sleep(2)

raw_df = pd.DataFrame(all_results)

print("\nSuccessful records:", len(raw_df))
print("Failed URLs:", len(failed_urls))

## 6. Initial data-quality checks

Before cleaning, check record count, missing values, and duplicate practice names/URLs.

In [ ]:
# Standardize empty cells for consistent quality checks.
for column in raw_df.columns:
    raw_df[column] = (
        raw_df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print("Shape:", raw_df.shape)

print("\nMissing/empty values:")
print((raw_df == "").sum())

print("\nDuplicate source URLs:",
      raw_df["source_url"].duplicated().sum())

print("Duplicate practice names:",
      raw_df["practice_name"].duplicated().sum())

## 7. Clean the source dataset

Two clinic URLs that did not provide usable clinic data were removed. Duplicate clinic records were then reviewed using the practice name, address, phone, and source URL.

The final source dataset is kept at **86 unique clinic records**.

In [ ]:
# Remove the two unusable clinic pages identified during source-data review.
unusable_urls = {
    "https://familydoctor.com.au/lilydale/",
    "https://familydoctor.com.au/mordialloc/"
}

clean_df = raw_df[
    ~raw_df["source_url"].isin(unusable_urls)
].copy()

# Remove exact duplicate clinic records while keeping the first verified record.
duplicate_key = ["practice_name", "full_address", "phone"]

clean_df = clean_df.drop_duplicates(
    subset=duplicate_key,
    keep="first"
).reset_index(drop=True)

# Keep only the fields required by the final dataset.
source_columns = [
    "practice_name",
    "phone",
    "full_address",
    "opening_hours",
    "services_raw",
    "source_url"
]

clean_df = clean_df[source_columns]

print("Final cleaned source records:", len(clean_df))
print("Duplicate practice names:",
      clean_df["practice_name"].duplicated().sum())
print("Duplicate source URLs:",
      clean_df["source_url"].duplicated().sum())

## 8. Save the cleaned source data

This file is the source-of-truth dataset before AI enrichment.

In [ ]:
clean_source_path = "Healthcare_Data_Cleaned.csv"

clean_df.to_csv(
    clean_source_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", clean_source_path)

## 9. AI enrichment

The AI enrichment uses the final evidence-based Prompt V2 documented separately in the project.

Only two AI-generated fields are required in the final dataset:

- `service_categories`
- `telehealth`

### Core AI rules

1. Use only the current clinic record.
2. Use `services_raw` as the source for service categories.
3. Do not add services based on general medical knowledge.
4. Do not copy information from another clinic.
5. Set `telehealth` to **Yes** only when telehealth is explicitly mentioned.
6. Otherwise return **Unknown**.
7. Preserve the original source fields unchanged.

The AI-generated enrichment file is loaded below and merged back to the cleaned source data.

In [ ]:
# Load the AI enrichment result produced using Prompt V2.
ai_path = "Healthcare_AI_Enrichment_Final.csv"

ai_df = pd.read_csv(
    ai_path,
    encoding="utf-8-sig"
)

ai_df = ai_df[
    ["practice_name", "service_categories", "telehealth"]
].copy()

ai_df["practice_name"] = (
    ai_df["practice_name"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("AI enrichment records:", len(ai_df))
print("AI enrichment columns:", ai_df.columns.tolist())

## 10. Create the final Master Healthcare Dataset

The original source fields are combined with the two evidence-based AI enrichment fields.

In [ ]:
final_df = clean_df.merge(
    ai_df,
    on="practice_name",
    how="left",
    validate="one_to_one"
)

# Keep only the fields required for the final submission.
final_columns = [
    "practice_name",
    "phone",
    "full_address",
    "opening_hours",
    "services_raw",
    "service_categories",
    "telehealth",
    "source_url"
]

final_df = final_df[final_columns]

# Replace blank AI values with Unknown rather than inventing information.
final_df["telehealth"] = (
    final_df["telehealth"]
    .fillna("")
    .replace("", "Unknown")
)

print("Final dataset shape:", final_df.shape)
final_df.head()

## 11. Final quality validation

The final dataset must contain unique clinics, unique source URLs, the required columns, and no unnecessary enrichment fields.

In [ ]:
required_columns = [
    "practice_name",
    "phone",
    "full_address",
    "opening_hours",
    "services_raw",
    "service_categories",
    "telehealth",
    "source_url"
]

print("Records:", len(final_df))
print("Columns:", final_df.columns.tolist())
print("Required columns present:",
      set(required_columns).issubset(final_df.columns))
print("Duplicate practice names:",
      final_df["practice_name"].duplicated().sum())
print("Duplicate source URLs:",
      final_df["source_url"].duplicated().sum())

print("\nEmpty values by column:")
print((final_df == "").sum())

assert len(final_df) == 86
assert final_df["practice_name"].duplicated().sum() == 0
assert final_df["source_url"].duplicated().sum() == 0
assert final_df.columns.tolist() == required_columns

print("\n✓ Final quality checks passed.")

## 12. Export the final dataset

The final output contains only the required fields used in the assessment.

In [ ]:
final_csv = "Healthcare_Master_Final_Quality.csv"

final_df.to_csv(
    final_csv,
    index=False,
    encoding="utf-8-sig"
)

print(f"Final dataset saved: {final_csv}")
print(f"Records: {len(final_df)}")

## Final Result

**86 unique Australian medical practices** were processed.

Workflow:

**Family Doctor Directory → Selenium Scraping → Data Cleaning → Quality Validation → AI Enrichment → Final Master Healthcare Dataset**

The final dataset keeps the original source information and adds only the required evidence-based AI enrichment fields.

In [ ]:
# Close the Selenium browser after all scraping is complete.
driver.quit()
print("Browser closed.")